# 02 · Ingesta y entendimiento de NOAA Storm Events


In [1]:
#   %pip install pandas pyarrow

Importo las librerías que voy a utilizar


In [1]:
from pathlib import Path
import re

import pandas as pd

Defino las carpetas de datos originales y staging


In [2]:
RAW_DIR = Path("../data/raw/noaa/details")
STAGING_DIR = Path("../data/staging")

STAGING_DIR.mkdir(parents=True, exist_ok=True)

Busco los archivos descargados y controlo cantidad


In [3]:
files = sorted(RAW_DIR.glob("StormEvents_details-ftp_*.csv.gz"))

print("Cantidad de archivos encontrados:", len(files))

for file in files:
    print(file.name)

Cantidad de archivos encontrados: 16
StormEvents_details-ftp_v1.0_d2010_c20260323.csv.gz
StormEvents_details-ftp_v1.0_d2011_c20260323.csv.gz
StormEvents_details-ftp_v1.0_d2012_c20260323.csv.gz
StormEvents_details-ftp_v1.0_d2013_c20260323.csv.gz
StormEvents_details-ftp_v1.0_d2014_c20260323.csv.gz
StormEvents_details-ftp_v1.0_d2015_c20260323.csv.gz
StormEvents_details-ftp_v1.0_d2016_c20260323.csv.gz
StormEvents_details-ftp_v1.0_d2017_c20260519.csv.gz
StormEvents_details-ftp_v1.0_d2018_c20260323.csv.gz
StormEvents_details-ftp_v1.0_d2019_c20260323.csv.gz
StormEvents_details-ftp_v1.0_d2020_c20260323.csv.gz
StormEvents_details-ftp_v1.0_d2021_c20260323.csv.gz
StormEvents_details-ftp_v1.0_d2022_c20260625.csv.gz
StormEvents_details-ftp_v1.0_d2023_c20260323.csv.gz
StormEvents_details-ftp_v1.0_d2024_c20260728.csv.gz
StormEvents_details-ftp_v1.0_d2025_c20260728.csv.gz


Defino una función para extraer el año desde el nombre de cada archivo


In [4]:
def extract_year(filename):
    """
    Extrae el año desde el nombre del archivo NOAA.
    """

    match = re.search(
        r"_d(\d{4})_",
        filename)

    if match:
        return int(match.group(1))

    return None

Leo todos los archivos, normalizo sus columnas y agrego el año de procedencia


In [5]:
dataframes = []

for file in files:

    year = extract_year(file.name)

    print(f"Leyendo año {year}: {file.name}")

    df_year = pd.read_csv(file, low_memory=False)

    df_year.columns = (df_year.columns.str.strip().str.lower())

    df_year["source_year"] = year
    df_year["source_file"] = file.name

    dataframes.append(df_year)

Leyendo año 2010: StormEvents_details-ftp_v1.0_d2010_c20260323.csv.gz
Leyendo año 2011: StormEvents_details-ftp_v1.0_d2011_c20260323.csv.gz
Leyendo año 2012: StormEvents_details-ftp_v1.0_d2012_c20260323.csv.gz
Leyendo año 2013: StormEvents_details-ftp_v1.0_d2013_c20260323.csv.gz
Leyendo año 2014: StormEvents_details-ftp_v1.0_d2014_c20260323.csv.gz
Leyendo año 2015: StormEvents_details-ftp_v1.0_d2015_c20260323.csv.gz
Leyendo año 2016: StormEvents_details-ftp_v1.0_d2016_c20260323.csv.gz
Leyendo año 2017: StormEvents_details-ftp_v1.0_d2017_c20260519.csv.gz
Leyendo año 2018: StormEvents_details-ftp_v1.0_d2018_c20260323.csv.gz
Leyendo año 2019: StormEvents_details-ftp_v1.0_d2019_c20260323.csv.gz
Leyendo año 2020: StormEvents_details-ftp_v1.0_d2020_c20260323.csv.gz
Leyendo año 2021: StormEvents_details-ftp_v1.0_d2021_c20260323.csv.gz
Leyendo año 2022: StormEvents_details-ftp_v1.0_d2022_c20260625.csv.gz
Leyendo año 2023: StormEvents_details-ftp_v1.0_d2023_c20260323.csv.gz
Leyendo año 2024: St

Uno todos los años en una sola tabla


In [6]:
df_storm = pd.concat(dataframes, ignore_index=True)

print("Filas:", df_storm.shape[0])
print("Columnas:", df_storm.shape[1])

Filas: 1037691
Columnas: 53


In [7]:
del dataframes

Reviso las primeras filas de la base consolidada


In [8]:
df_storm.head()

,begin_yearmonth,begin_day,begin_time,end_yearmonth,end_day,end_time,episode_id,event_id,state,state_fips,...,end_location,begin_lat,begin_lon,end_lat,end_lon,episode_narrative,event_narrative,data_source,source_year,source_file
0,201011,22,1531,201011,22,1532,46247,268262,ILLINOIS,17,...,BIG FOOT,42.4932,-88.5704,42.4951,-88.5695,Strong to severe thunderstorms moved across pa...,A tornado touched down in the extreme northern...,CSV,2010,StormEvents_details-ftp_v1.0_d2010_c20260323.c...
1,201007,7,1251,201007,7,1630,43850,254780,NEW HAMPSHIRE,33,...,NaN,NaN,NaN,NaN,NaN,A strong ridge built into Southern New England...,Heat index values at the Nashua Boire Field (K...,CSV,2010,StormEvents_details-ftp_v1.0_d2010_c20260323.c...
2,201001,17,2300,201001,18,1500,36500,211550,NEW HAMPSHIRE,33,...,NaN,NaN,NaN,NaN,NaN,A coastal storm passing southern New England j...,Four to eight inches fell across eastern Hills...,CSV,2010,StormEvents_details-ftp_v1.0_d2010_c20260323.c...
3,201010,1,830,201010,1,1000,44854,260014,NEW HAMPSHIRE,33,...,NaN,NaN,NaN,NaN,NaN,Several waves of low pressure moved across Sou...,"In Manchester, firefighters responded to about...",CSV,2010,StormEvents_details-ftp_v1.0_d2010_c20260323.c...
4,201007,6,951,201007,6,1830,43850,254779,NEW HAMPSHIRE,33,...,NaN,NaN,NaN,NaN,NaN,A strong ridge built into Southern New England...,Heat index values at the Manchester Airport (K...,CSV,2010,StormEvents_details-ftp_v1.0_d2010_c20260323.c...


Reviso las columnas disponibles


In [9]:
df_storm.columns.tolist()

['begin_yearmonth',
 'begin_day',
 'begin_time',
 'end_yearmonth',
 'end_day',
 'end_time',
 'episode_id',
 'event_id',
 'state',
 'state_fips',
 'year',
 'month_name',
 'event_type',
 'cz_type',
 'cz_fips',
 'cz_name',
 'wfo',
 'begin_date_time',
 'cz_timezone',
 'end_date_time',
 'injuries_direct',
 'injuries_indirect',
 'deaths_direct',
 'deaths_indirect',
 'damage_property',
 'damage_crops',
 'source',
 'magnitude',
 'magnitude_type',
 'flood_cause',
 'category',
 'tor_f_scale',
 'tor_length',
 'tor_width',
 'tor_other_wfo',
 'tor_other_cz_state',
 'tor_other_cz_fips',
 'tor_other_cz_name',
 'begin_range',
 'begin_azimuth',
 'begin_location',
 'end_range',
 'end_azimuth',
 'end_location',
 'begin_lat',
 'begin_lon',
 'end_lat',
 'end_lon',
 'episode_narrative',
 'event_narrative',
 'data_source',
 'source_year',
 'source_file']

Compruebo cuántos registros hay por año


In [10]:
rows_by_year = (df_storm.groupby("source_year").size().reset_index(name="rows"))

rows_by_year

,source_year,rows
0,2010,62809
1,2011,79091
2,2012,64503
3,2013,59986
4,2014,59475
5,2015,57907
6,2016,56005
7,2017,57041
8,2018,62699
9,2019,67864


Compruebo que el año informado coincida con el año del archivo de origen


In [11]:
year_comparison = (df_storm["year"].eq(df_storm["source_year"]).value_counts(dropna=False))

year_comparison

True    1037691
Name: count, dtype: int64

Reviso los tipos de datos y valores no nulos


In [12]:
df_storm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1037691 entries, 0 to 1037690
Data columns (total 53 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   begin_yearmonth     1037691 non-null  int64  
 1   begin_day           1037691 non-null  int64  
 2   begin_time          1037691 non-null  int64  
 3   end_yearmonth       1037691 non-null  int64  
 4   end_day             1037691 non-null  int64  
 5   end_time            1037691 non-null  int64  
 6   episode_id          1037691 non-null  int64  
 7   event_id            1037691 non-null  int64  
 8   state               1037691 non-null  object 
 9   state_fips          1037691 non-null  int64  
 10  year                1037691 non-null  int64  
 11  month_name          1037691 non-null  object 
 12  event_type          1037691 non-null  object 
 13  cz_type             1037691 non-null  object 
 14  cz_fips             1037691 non-null  int64  
 15  cz_name        

Compruebo la memoria ocupada por la base consolidada


In [13]:
memory_mb = (df_storm.memory_usage(deep=True).sum()/ 1024**2)

print(f"Memoria utilizada: {memory_mb:,.2f} MB")

Memoria utilizada: 2,221.13 MB


Compruebo si existen filas duplicadas


In [14]:
df_storm.duplicated().sum()

0

Comparo la cantidad de filas con la cantidad de eventos únicos


In [15]:
len(df_storm)

1037691

In [16]:
df_storm["event_id"].nunique()

1037691

Reviso los valores nulos de todas las variables


In [17]:
missing_report = pd.DataFrame({
    "missing_rows": df_storm.isna().sum(),
    "missing_pct": (df_storm.isna().mean() * 100).round(2)})

missing_report = (missing_report.sort_values("missing_pct", ascending=False))

missing_report.head(30)

,missing_rows,missing_pct
category,1037253,99.96
tor_other_wfo,1034501,99.69
tor_other_cz_state,1034501,99.69
tor_other_cz_fips,1034507,99.69
tor_other_cz_name,1034507,99.69
tor_f_scale,1014502,97.77
tor_length,1014502,97.77
tor_width,1014502,97.77
flood_cause,930296,89.65
magnitude_type,653271,62.95


Reviso los nulos de las variables principales


In [19]:
main_columns = ["event_id","episode_id","state","year","month_name","event_type",
    "begin_date_time","end_date_time","source","magnitude","damage_property",
    "damage_crops","injuries_direct","injuries_indirect","deaths_direct",
    "deaths_indirect","source_year"]

df_storm[main_columns].head(10)

,event_id,episode_id,state,year,month_name,event_type,begin_date_time,end_date_time,source,magnitude,damage_property,damage_crops,injuries_direct,injuries_indirect,deaths_direct,deaths_indirect,source_year
0,268262,46247,ILLINOIS,2010,November,Tornado,22-NOV-10 15:31:00,22-NOV-10 15:32:00,NWS Storm Survey,NaN,0,0,0,0,0,0,2010
1,254780,43850,NEW HAMPSHIRE,2010,July,Heat,07-JUL-10 12:51:00,07-JUL-10 16:30:00,AWOS,NaN,0.00K,0.00K,0,0,0,0,2010
2,211550,36500,NEW HAMPSHIRE,2010,January,Heavy Snow,17-JAN-10 23:00:00,18-JAN-10 15:00:00,CoCoRaHS,NaN,0.00K,0.00K,0,0,0,0,2010
3,260014,44854,NEW HAMPSHIRE,2010,October,Strong Wind,01-OCT-10 08:30:00,01-OCT-10 10:00:00,Newspaper,45.0,50.00K,0.00K,0,0,0,0,2010
4,254779,43850,NEW HAMPSHIRE,2010,July,Heat,06-JUL-10 09:51:00,06-JUL-10 18:30:00,ASOS,NaN,0.00K,0.00K,0,0,0,0,2010
5,273769,46989,NEW HAMPSHIRE,2010,December,Winter Storm,26-DEC-10 17:00:00,27-DEC-10 18:00:00,Trained Spotter,NaN,0.00K,0.00K,0,0,0,0,2010
6,215305,37004,NEW HAMPSHIRE,2010,February,High Wind,25-FEB-10 23:05:00,26-FEB-10 00:38:00,ASOS,55.0,2.50M,0.00K,0,0,0,0,2010
7,214918,36944,NEW HAMPSHIRE,2010,February,Heavy Snow,16-FEB-10 12:00:00,17-FEB-10 00:00:00,Trained Spotter,NaN,0.00K,0.00K,0,0,0,0,2010
8,218778,37397,NEW HAMPSHIRE,2010,March,Strong Wind,14-MAR-10 13:45:00,14-MAR-10 14:15:00,Newspaper,35.0,10.00K,0.00K,2,0,1,0,2010
9,1244226,46247,ILLINOIS,2010,November,Tornado,22-NOV-10 15:00:00,22-NOV-10 15:03:00,NWS Storm Survey,NaN,500000,0,6,0,0,0,2010


Compruebo la frecuencia de cada tipo de evento


In [18]:
event_type_counts = (df_storm["event_type"].value_counts(dropna=False).reset_index())

event_type_counts.columns = ["event_type", "rows"]

event_type_counts.head(20)

,event_type,rows
0,Thunderstorm Wind,276713
1,Hail,158317
2,Flash Flood,63384
3,Winter Weather,60774
4,Drought,57212
5,High Wind,56429
6,Winter Storm,47412
7,Flood,42252
8,Heavy Snow,36267
9,Marine Thunderstorm Wind,33491


Reviso la cantidad de categorías de las variables principales


In [19]:
categorical_columns = ["state", "month_name", "event_type", "source", "cz_type", "cz_name"]

for column in categorical_columns:

    unique_values = (df_storm[column].nunique(dropna=True))

    print(f"{column}: {unique_values} categorías")

state: 69 categorías
month_name: 12 categorías
event_type: 54 categorías
source: 44 categorías
cz_type: 2 categorías
cz_name: 4732 categorías


Compruebo la cantidad de registros por estado


In [20]:
state_counts = (df_storm["state"].value_counts(dropna=False).reset_index())

state_counts.columns = ["state","rows"]

state_counts.head(20)

,state,rows
0,TEXAS,76632
1,KANSAS,39246
2,MISSOURI,34591
3,VIRGINIA,34032
4,IOWA,33618
5,OKLAHOMA,33439
6,ILLINOIS,33124
7,NEBRASKA,31444
8,SOUTH DAKOTA,31256
9,KENTUCKY,31210


Reviso los formatos registrados en las variables de daños


In [21]:
df_storm["damage_property"].value_counts(dropna=False).head(20)

damage_property
0.00K      611911
NaN        204143
1.00K       34068
5.00K       26835
10.00K      22523
2.00K       20920
3.00K       11535
50.00K       9244
0.50K        8889
25.00K       8690
20.00K       8635
15.00K       8142
100.00K      6221
4.00K        3985
30.00K       3812
8.00K        2456
75.00K       2225
250.00K      2173
500.00K      2080
200.00K      1984
Name: count, dtype: int64

In [22]:
df_storm["damage_crops"].value_counts(dropna=False).head(20)

damage_crops
0.00K      814725
NaN        206757
1.00K        2382
5.00K        1316
10.00K       1168
2.00K         965
0.10K         860
50.00K        707
3.00K         644
0.50K         508
100.00K       500
500.00K       479
250.00K       405
0.25K         391
20.00K        377
1.00M         347
25.00K        308
15.00K        255
0.01K         240
30.00K        198
Name: count, dtype: int64

Reviso ejemplos de eventos con daños registrados


In [23]:
damage_examples = df_storm.loc[(df_storm["damage_property"].notna() | df_storm["damage_crops"].notna()),
    ["event_id", "state", "event_type", "damage_property", "damage_crops", "injuries_direct", "deaths_direct"]]

damage_examples.head(20)

,event_id,state,event_type,damage_property,damage_crops,injuries_direct,deaths_direct
0,268262,ILLINOIS,Tornado,0,0,0,0
1,254780,NEW HAMPSHIRE,Heat,0.00K,0.00K,0,0
2,211550,NEW HAMPSHIRE,Heavy Snow,0.00K,0.00K,0,0
3,260014,NEW HAMPSHIRE,Strong Wind,50.00K,0.00K,0,0
4,254779,NEW HAMPSHIRE,Heat,0.00K,0.00K,0,0
5,273769,NEW HAMPSHIRE,Winter Storm,0.00K,0.00K,0,0
6,215305,NEW HAMPSHIRE,High Wind,2.50M,0.00K,0,0
7,214918,NEW HAMPSHIRE,Heavy Snow,0.00K,0.00K,0,0
8,218778,NEW HAMPSHIRE,Strong Wind,10.00K,0.00K,2,1
9,1244226,ILLINOIS,Tornado,500000,0,6,0


Obtengo estadísticas descriptivas de las variables numéricas


In [24]:
numeric_columns = ["magnitude", "injuries_direct", "injuries_indirect", "deaths_direct", "deaths_indirect", "begin_lat", "begin_lon", "end_lat", "end_lon" ]

df_storm[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
magnitude,543207.0,37.163155,24.145976,0.0000,1.75000,50.0000,52.000000,174.0000
injuries_direct,1037691.0,0.035006,2.351918,0.0000,0.00000,0.0000,0.000000,1150.0000
injuries_indirect,1037691.0,0.009388,0.750408,0.0000,0.00000,0.0000,0.000000,500.0000
deaths_direct,1037691.0,0.010399,0.344821,0.0000,0.00000,0.0000,0.000000,158.0000
deaths_indirect,1037691.0,0.003688,0.124308,0.0000,0.00000,0.0000,0.000000,56.0000
begin_lat,636046.0,37.765415,5.140898,-14.4000,34.50000,38.2000,41.290000,70.5029
begin_lon,636046.0,-90.068658,11.538747,-171.0327,-97.15000,-89.2800,-81.510000,171.4689
end_lat,636046.0,37.763801,5.141634,-14.4560,34.50000,38.2000,41.290000,70.2789
end_lon,636046.0,-90.062125,11.536632,-170.9059,-97.14375,-89.2682,-81.501725,171.4689


Reviso el formato de las fechas de inicio y fin


In [25]:
df_storm[["begin_date_time","end_date_time"]].head(10)

,begin_date_time,end_date_time
0,22-NOV-10 15:31:00,22-NOV-10 15:32:00
1,07-JUL-10 12:51:00,07-JUL-10 16:30:00
2,17-JAN-10 23:00:00,18-JAN-10 15:00:00
3,01-OCT-10 08:30:00,01-OCT-10 10:00:00
4,06-JUL-10 09:51:00,06-JUL-10 18:30:00
5,26-DEC-10 17:00:00,27-DEC-10 18:00:00
6,25-FEB-10 23:05:00,26-FEB-10 00:38:00
7,16-FEB-10 12:00:00,17-FEB-10 00:00:00
8,14-MAR-10 13:45:00,14-MAR-10 14:15:00
9,22-NOV-10 15:00:00,22-NOV-10 15:03:00


Guardo la base consolidada en formato Parquet


In [28]:
STAGING_FILE = (STAGING_DIR / "storm_events_details_2010_2025.parquet")

df_storm.to_parquet(STAGING_FILE, index=False)

print("Archivo guardado")

Archivo guardado
